# 41 · Comparación de los demás codificadores radiográficos

**Revisión 2.0 · Plan metodológico v1.2 · GPU T4.**

La campaña contiene **25 trabajos**; se ejecutan solo los pendientes, en este orden: **ViT-B/16, ConvNeXtV2-Tiny, Skelex, BiomedCLIP y MambaOut-Tiny**; cinco pliegues por codificador, con semilla 2026. Los codificadores permanecen congelados: solo se entrena la cabeza de predicción. Cada pliegue nuevo comienza desde el checkpoint externo registrado. Si existe un guardado compatible de una época de ese mismo pliegue, se restaura sin reutilizar pesos de otro pliegue.

Se mantienen el conjunto de desarrollo, las particiones por participante, las entradas de 224 × 224 y el aumento gamma v1.2 únicamente en entrenamiento. La validación conserva su transformación determinista y cada codificador utiliza su normalización registrada.

Se necesitan los registros completos de las libretas **33 y 38** ya guardados en Drive. Se autentican automáticamente; no tienes que volver a ejecutarlas. Revisa las tres vistas del aumento de la libreta 38 antes de iniciar la campaña. Al ejecutar la celda 4, se comprobarán los pesos y un forward sintético en GPU únicamente de los codificadores pendientes. Se recuperan las pruebas breves compatibles ya guardadas; solo si faltan se ejecutan dos actualizaciones en el pliegue 0 de desarrollo para verificar gradientes y recursos con la receta aprobada. Estos cinco controles se registran aparte, no participan en la selección del modelo y sus pesos no se reutilizan en los 25 trabajos completos.

Cada trabajo completo se guarda y verifica en Drive antes de continuar. Una reconexión recupera los trabajos completos compatibles guardados por la revisión anterior o esta revisión, sin copiar sus pesos. Los trabajos nuevos guardan al final de cada época el estado necesario para retomarse después de un corte. Una época interrumpida antes del guardado debe repetirse. El progreso dentro de un pliegue antiguo no guardado no se puede recuperar. Ejecuta una sola sesión de esta campaña.

DenseNet121 ya tiene su campaña en la libreta 40 y no se vuelve a entrenar aquí. DINOv3 permanece fuera por acceso pendiente. Esta libreta no descongela codificadores, no ejecuta fusión multimodal, calibración, selección de umbral ni ajuste final. Los originales y las particiones se conservan; la prueba reservada permanece cerrada. **Esta campaña ya fue autorizada. El entrenamiento comienza al ejecutar la celda 4; no tienes que escribir rutas ni repetir una frase de autorización.**

In [ ]:
from pathlib import Path, PurePosixPath
import json, os, re, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile

CODE_REVISION = '7aefc0a92495ecce287e522820932ca2b5a3182f'
if not re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION) or CODE_REVISION == '0' * 40:
    raise RuntimeError('Esta copia no tiene una revisión publicada. Usa la libreta final entregada.')
if os.environ.get('COLAB_GPU', '0') != '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución. No uses CPU ni TPU.')
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Esta libreta debe ejecutarse personalmente en Google Colab; no es un ensayo local.')
if 'knee' in sys.modules or any(name.startswith('knee.') for name in sys.modules):
    raise RuntimeError('Reinicia el entorno para cargar la revisión fijada de esta libreta.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_image_screening41_'))
REPO_DIR = SESSION_ROOT / 'repository'
archive_path = SESSION_ROOT / 'code.zip'
url = 'https://codeload.github.com/AlonsoYess/Knee/zip/' + CODE_REVISION
with urllib.request.urlopen(url, timeout=60) as response, archive_path.open('xb') as target:
    if response.geturl() != url:
        raise RuntimeError('La descarga del código cambió de ubicación.')
    total = 0
    while True:
        chunk = response.read(1024**2)
        if not chunk:
            break
        total += len(chunk)
        if total > 128 * 1024**2:
            raise RuntimeError('El archivo del repositorio supera el límite autorizado.')
        target.write(chunk)
REPO_DIR.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if not members or len(members) > 20000 or sum(i.file_size for i in members) > 512 * 1024**2:
        raise RuntimeError('El contenido del repositorio supera los límites de extracción.')
    seen = set()
    for item in members:
        name = item.filename.rstrip('/')
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or '\0' in name or any(p in ('', '.', '..') for p in parts) or parts[0] != 'Knee-' + CODE_REVISION or name in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Se encontró una ruta no permitida en el repositorio.')
        seen.add(name)
        destination = REPO_DIR.joinpath(*parts[1:])
        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            if len(parts) < 2:
                raise RuntimeError('El archivo del repositorio no tiene una ruta relativa válida.')
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024**2)
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee import image_campaign_recovery as orchestration

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode:
        detail = re.sub(r'https?://\S+', '<url>', (result.stdout + result.stderr)[-12000:])
        detail = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', detail)
        print(detail)
        raise RuntimeError('La comprobación previa de la campaña se detuvo. Conserva el error; no inicies entrenamiento.')
    return result.stdout

print('Revisión exacta del código preparada. No se abrieron datos de participantes.')

CAMPAIGN = 'remaining_encoders_frozen'


## 1. Preparar el intérprete GPU aislado

Se conservan las dependencias fijadas y se verifica el recurso seleccionado. Esta celda no inicia entrenamiento.

In [ ]:
print('Preparando el intérprete aislado; todavía no se inició entrenamiento…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/train_image_screening_colab.py'), '--operation', 'setup', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--destination', str(SESSION_ROOT / 'isolated_runtime')]))
NEURAL_PYTHON = Path(runtime['python_path'])
RUN_ENV = orchestration.runtime_environment(REPO_DIR, CAMPAIGN)
print('Intérprete preparado. Todavía no se entrenó ningún trabajo.')


## 2. Comprobar la implementación con datos sintéticos

Se comprueban la lista y el orden de los 25 trabajos, las autorizaciones, el aumento, los archivos de mayor tamaño y la recuperación. Estas pruebas calculan en CPU con la compilación de PyTorch para GPU; la GPU queda oculta en este paso. No se entrenan datos de participantes.

In [ ]:
TEST_ENV = dict(RUN_ENV)
TEST_ENV.update(CUDA_VISIBLE_DEVICES='', COLAB_GPU='0')
CPU_TESTS = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_image_screening_colab.py'), '--operation', 'tests', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR)], cwd=REPO_DIR, env=TEST_ENV))
print(f"Pruebas sintéticas aprobadas: {CPU_TESTS['tests_run']}; omitidas: {CPU_TESTS['skipped']}.")


## 3. Localizar las comprobaciones previas y reconstruir desarrollo

Las carpetas de registros privados y desarrollo se leen de la configuración privada guardada en Drive. No es necesario escribir rutas. Si esa configuración no es válida, la celda se detiene sin buscar otras carpetas. Se autentican las comprobaciones 33 y 38 y la integridad del paquete de desarrollo. Esta celda no descarga ni abre el paquete reservado de prueba.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MY_DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()
PROJECT_ROOT = orchestration.checked_path(os.environ.get('KNEE_DATA_ROOT') or MY_DRIVE_ROOT / 'KneeOAI').resolve()
if not PROJECT_ROOT.is_dir() or not PROJECT_ROOT.is_relative_to(MY_DRIVE_ROOT):
    raise RuntimeError('No se encontró la carpeta privada configurada dentro de tu Drive montado.')
SETTINGS_PATH = orchestration.checked_path(PROJECT_ROOT / 'colab_runtime_paths.json')
if not SETTINGS_PATH.is_file() or not 0 < SETTINGS_PATH.stat().st_size <= 16384:
    raise RuntimeError('Falta la configuración privada de las carpetas de Colab.')
settings = orchestration._json(SETTINGS_PATH)
if not isinstance(settings, dict) or set(settings) != {'version', 'records_relative', 'development_relative'} or settings.get('version') != 'knee_colab_runtime_paths_v1.0':
    raise RuntimeError('La configuración privada de carpetas no tiene el formato esperado.')
selected_paths = []
for key in ('records_relative', 'development_relative'):
    relative = settings[key]
    if not isinstance(relative, str) or not relative or len(relative) > 1024 or '\\' in relative or ':' in relative:
        raise RuntimeError('La configuración debe contener rutas relativas válidas.')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or not parts.parts or any(part in ('', '.', '..') for part in relative.split('/')):
        raise RuntimeError('La configuración no puede salir de la carpeta del proyecto.')
    selected = orchestration.checked_path(PROJECT_ROOT.joinpath(*parts.parts)).resolve()
    if not selected.is_dir() or not selected.is_relative_to(PROJECT_ROOT):
        raise RuntimeError('Una de las carpetas privadas configuradas no existe dentro del proyecto.')
    selected_paths.append(selected)
PERSISTENT_ROOT, REMOTE_PACKAGE_DIR = selected_paths
if PERSISTENT_ROOT.is_relative_to(REMOTE_PACKAGE_DIR) or REMOTE_PACKAGE_DIR.is_relative_to(PERSISTENT_ROOT):
    raise RuntimeError('Las carpetas de registros y desarrollo deben estar separadas.')
if not orchestration.checked_path(REMOTE_PACKAGE_DIR / 'transport.json').is_file():
    raise RuntimeError('La carpeta de desarrollo configurada no contiene transport.json.')
print('Carpetas privadas localizadas automáticamente. No se inició entrenamiento.')
proof = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_image_screening_colab.py'), '--operation', 'locate', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--records-root', str(PERSISTENT_ROOT)], cwd=REPO_DIR, env=RUN_ENV))
PREPARATION_DIR = Path(proof['preparation_dir'])
LOCAL_PACKAGE_DIR = SESSION_ROOT / 'development'
REFERENCE_DIR = SESSION_ROOT / 'reference'
REFERENCE_DIR.mkdir(exist_ok=False)
LOCAL_REFERENCE = REFERENCE_DIR / 'model_inputs_loading_reference_v1.json'
transport = orchestration.stage_development(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR, LOCAL_REFERENCE)
print('Comprobación 38 localizada. Paquete de desarrollo reconstruido y verificado.')


## 4. Iniciar la campaña autorizada: cinco pruebas breves y 25 trabajos

Orden: **ViT-B/16 → ConvNeXtV2-Tiny → Skelex → BiomedCLIP → MambaOut-Tiny**. Se reconocen y omiten los trabajos completos guardados. Solo se verifican en GPU los codificadores pendientes, reutilizando sus pruebas breves ya guardadas cuando correspondan. Después se completan únicamente los pliegues pendientes, siempre con el codificador congelado y aumento gamma v1.2. Un trabajo pendiente retoma su última época guardada si existe; de lo contrario comienza desde los pesos externos registrados. Los controles breves no aportan pesos ni métricas a esa comparación. El progreso identifica modelo y pliegue. Los checkpoints grandes requieren espacio local y tiempo de copia a Drive. Cada trabajo se guarda y verifica antes de continuar.

**Al ejecutar esta celda se retoma la campaña que ya autorizaste; no se solicita otra frase.** La autorización se limita a los cinco codificadores y los cinco pliegues descritos. Si Colab se desconecta, vuelve a abrir esta misma revisión y ejecuta las celdas: se recuperan los trabajos completos compatibles.

In [ ]:
if CAMPAIGN != 'remaining_encoders_frozen':
    raise RuntimeError('Esta libreta no autoriza una campaña distinta.')
import queue, threading, time, uuid

def training_run(arguments, *, cwd, env):
    # Forward actual stderr progress; stdout contains the final structured record.
    process = subprocess.Popen(arguments, cwd=cwd, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    messages = queue.Queue()
    def forward(stream, channel):
        try:
            for line in iter(stream.readline, ''):
                messages.put((channel, line))
        finally:
            stream.close()
            messages.put((channel, None))
    for stream, channel in ((process.stdout, 'stdout'), (process.stderr, 'stderr')):
        threading.Thread(target=forward, args=(stream, channel), daemon=True).start()
    started, last_heartbeat, closed = time.monotonic(), time.monotonic(), set()
    stdout_lines = []
    try:
        while len(closed) < 2:
            try:
                channel, line = messages.get(timeout=1)
            except queue.Empty:
                channel, line = None, None
            if channel is not None:
                if line is None:
                    closed.add(channel)
                elif channel == 'stdout':
                    stdout_lines.append(line)
                else:
                    safe = re.sub(r'https?://\S+', '<url>', line.rstrip())
                    safe = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', safe)
                    print(safe, flush=True)
            now = time.monotonic()
            if now - last_heartbeat >= 30 and process.poll() is None:
                print(f'Subproceso activo; tiempo transcurrido: {int(now - started)} s. No es un porcentaje de avance.', flush=True)
                last_heartbeat = now
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if status:
        raise RuntimeError('El entrenamiento se detuvo. Conserva el error y los trabajos completos guardados en Drive; no cambies la receta.')
    return ''.join(stdout_lines)

LOCAL_OUTPUT = SESSION_ROOT / ('methodology_campaign_' + uuid.uuid4().hex)
TEST_RECEIPT = SESSION_ROOT / 'synthetic_test_receipt.json'
RUNTIME_RECEIPT = SESSION_ROOT / 'setup_receipt.json'
for receipt_path, payload in ((TEST_RECEIPT, CPU_TESTS), (RUNTIME_RECEIPT, runtime)):
    if receipt_path.exists():
        if orchestration._json(receipt_path) != payload:
            raise RuntimeError('El recibo previo pertenece a otra preparación; inicia una sesión nueva.')
    else:
        orchestration._save(receipt_path, payload)
arguments = [str(NEURAL_PYTHON), '-u', '-B', str(REPO_DIR / 'scripts/train_image_screening_colab.py'), '--operation', 'train', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--preparation-dir', str(PREPARATION_DIR), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-path', str(LOCAL_REFERENCE), '--output-dir', str(LOCAL_OUTPUT), '--records-root', str(PERSISTENT_ROOT), '--code-revision', CODE_REVISION, '--test-receipt', str(TEST_RECEIPT), '--runtime-receipt', str(RUNTIME_RECEIPT), '--approval-statement', orchestration.APPROVAL_STATEMENTS[CAMPAIGN]]
if CAMPAIGN == 'remaining_encoders_frozen':
    WEIGHTS_ROOT = SESSION_ROOT / 'verified_public_weights'
    WEIGHTS_ROOT.mkdir(exist_ok=True)
    arguments.extend(['--weights-root', str(WEIGHTS_ROOT)])
print('Buscando trabajos completos para omitirlos; se entrenarán solo los pendientes…', flush=True)
training_log = training_run(arguments, cwd=REPO_DIR, env=RUN_ENV)
FINAL_CHECK = json.loads((LOCAL_OUTPUT / orchestration.SUMMARY_FILENAME).read_text(encoding='utf-8'))
orchestration.validate_result(FINAL_CHECK)
print('Campaña terminada. Resultados privados guardados y verificados en Drive.')


## 5. Mostrar el resultado

La selección utiliza la media no ponderada de AP de los cinco pliegues de cada candidato. La AP al reunir las predicciones fuera de pliegue se informa de forma descriptiva. Son resultados de desarrollo; la evaluación final se realizará posteriormente.

Comparte únicamente el **JSON final**. Las predicciones individuales y los modelos permanecen privados.

In [ ]:
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
print('Resultados de desarrollo guardados. No se abrió la prueba reservada ni se ajustó la calibración final.')
